<a href="https://colab.research.google.com/github/ArseniyFaryna/Intelligent_analisis_labs/blob/main/%D0%9B%D0%B0%D0%B13_%D0%B7%D0%B0%D0%B2%D0%B4%D0%B0%D0%BD%D0%BD%D1%8F_%D1%81%D1%82%D1%83%D0%B4%D0%B5%D0%BD%D1%82%D1%83.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [3]:
ПРІЗВИЩЕ_ІМЯ = 'Фарина Арсеній'      # напишіть тут своє прізвище та ім'я
ГРУПА        = 'ШІ-31'
НОМЕР        = 18       # ваш номер за списком групи (1–30)

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [4]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 18   ·   Фарина Арсеній   ·   ШІ-31

Ваші вісім стовпців:
   1. номер і/х
   2. Стать
   3. ступінь КТ
   4. Дата розтину
   5. spO2 без дот
   6. Фібрин1
   7. СРБ2
   8. ФЕРИТ1

Ваша пара для аналізу зв’язку: Гемогл1 та Гематокрит1

Ваше додаткове питання:
   Знайдіть у файлі стовпець, де одна й та сама величина записана у двох несумісних кодуваннях. Скільки спостережень втратить наївне обчислення середнього?


In [5]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,номер і/х,Стать,ступінь КТ,Дата розтину,spO2 без дот,Фібрин1,СРБ2,ФЕРИТ1
0,2377/239,ч,сердній,2022-03-11,90.0,4,6.0,NaN
1,1121/19,ч,середній,NaT,83.0,5,1.0,839.0
2,283/674,ж,критичний,NaT,78.0,5.29,6.0,NaN
3,2372/513,ж,критичний,NaT,82.0,2.5,6.0,NaN
4,1441/671,ж,NaN,NaT,80.0,4.29,9.0,NaN
5,2573/354,ж,середній,2021-12-26,72.0,5,6.0,NaN
6,1762/555,ч,важкий,NaT,89.0,3.24,6.0,NaN
7,2664/348,ж,легкий,NaT,90.0,5.04,6.0,NaN


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [6]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   номер і/х     121 non-null    object        
 1   Стать         121 non-null    object        
 2   ступінь КТ    42 non-null     object        
 3   Дата розтину  61 non-null     datetime64[ns]
 4   spO2 без дот  120 non-null    float64       
 5   Фібрин1       115 non-null    object        
 6   СРБ2          83 non-null     float64       
 7   ФЕРИТ1        2 non-null      float64       
dtypes: datetime64[ns](1), float64(3), object(4)
memory usage: 7.7+ KB


In [7]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- номер і/х
    непорожніх 121 з 121 | унікальних 119 | типи: int×65, str×56
    приклади: ["'2377/239'", "'1121/19'", "'283/674'", "'2372/513'"]

--- Стать
    непорожніх 121 з 121 | унікальних 4 | типи: str×121
    приклади: ["'ч'", "'ж'", "'чол'", "'жін'"]

--- ступінь КТ
    непорожніх 42 з 121 | унікальних 12 | типи: str×42
    приклади: ["'сердній'", "'середній'", "'критичний'", "'важкий'"]

--- Дата розтину
    непорожніх 61 з 121 | унікальних 50 | типи: Timestamp×61
    приклади: ["Timestamp('2022-03-11 00:00:00')", "Timestamp('2021-12-26 00:00:00')", "Timestamp('2021-11-19 00:00:00')", "Timestamp('2022-05-13 00:00:00')"]

--- spO2 без дот
    непорожніх 120 з 121 | унікальних 33 | типи: float×120
    приклади: ['np.float64(90.0)', 'np.float64(83.0)', 'np.float64(78.0)', 'np.float64(82.0)']

--- Фібрин1
    непорожніх 115 з 121 | унікальних 87 | типи: float×93, int×21, str×1
    приклади: ['4', '5', '5.29', '2.5']

--- СРБ2
    непорожніх 83 з 121 | унікальних 17 | типи: floa

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [8]:
ПАСПОРТ = {
    # 'назва стовпця': ('шкала', 'обґрунтування однією фразою'),
    'номер і/х'     : ('номінальна', 'Значення в цій колонці текстові, їх можна перевіряти виключно на рівність чи відмінність'),
    'Стать'         : ('номінальна', 'Дані в цій колонці можна перевіряти на рівність чи не рівність, тобто жінка чи чоловік'),
    'ступінь КТ'    : ('порядкова', 'Категорії мають чіткий природний порядок наростання тяжкості ураження легень, проте точна числова відстань між сусідніми рівнями не визначена'),
    'Дата розтину'  : ('інтервальна', 'Різниця між двома датами має чітки природний порядок, але точна числова відстань між датами не визначена'),
    'spO2 без дот'  : ('абсолютна', 'Показник має природний нуль - це повна відсутність кисню та змістовні пропорції, а через безрозмірність відсотків його можна класифікувати як абсолютну шкалу'),
    'Фібрин1'       : ('відношень', 'Фізична концентрація речовини із природним нулем - це повна відсутність молекул біомаркера, для яких операції віднімання та ділення мають прямий зміст, а числове значення залежить від обраної розмірності'),
    'СРБ2'          : ('відношень', 'Фізична концентрація речовини із природним нулем - це повна відсутність молекул біомаркера, для яких операції віднімання та ділення мають прямий зміст, а числове значення залежить від обраної розмірності'),
    'ФЕРИТ1'        : ('відношень', 'Фізична концентрація речовини із природним нулем - це повна відсутність молекул біомаркера, для яких операції віднімання та ділення мають прямий зміст, а числове значення залежить від обраної розмірності'),
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,номер і/х,номінальна,"Значення в цій колонці текстові, їх можна перевіряти вик..."
1,Стать,номінальна,Дані в цій колонці можна перевіряти на рівність чи не рі...
2,ступінь КТ,порядкова,Категорії мають чіткий природний порядок наростання тяжк...
3,Дата розтину,інтервальна,"Різниця між двома датами має чітки природний порядок, ал..."
4,spO2 без дот,абсолютна,Показник має природний нуль - це повна відсутність кисню...
5,Фібрин1,відношень,Фізична концентрація речовини із природним нулем - це по...
6,СРБ2,відношень,Фізична концентрація речовини із природним нулем - це по...
7,ФЕРИТ1,відношень,Фізична концентрація речовини із природним нулем - це по...


In [9]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [10]:
def допустима_статистика(стовпець, шкала):
    """Повертає словник статистик, дозволених для цієї шкали."""
    з = df[стовпець].dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()

    if ш in ('ідентифікатор',):
        return р
    # ваш код: додайте моду й частоти для всіх шкал
    # для порядкової — медіану та квантилі
    # для інтервальної та вище — середнє й стандартне відхилення
    # для відношень і абсолютної — коефіцієнт варіації
    р['мода'] = з.mode().tolist()
    р['частоти'] = з.value_counts().to_dict()

    if ш in ('порядкова', 'інтервальна', 'відношень', 'абсолютна'):
      try:
        р['медіана'] = з.median()
        р['квантиль_0.25'] = з.quantile(0.25)
        р['квантиль_0.75'] = з.quantile(0.75)
      except Exception:
        pass

    if ш in ('інтервальна', 'відношень', 'абсолютна'):
      try:
        р['середнє'] = з.mean()
        р['ст_відхилення'] = з.std()
      except Exception:
        pass

    if ш in ('відношень', 'абсолютна'):
      try:
        if р.get('середнє') and р['середнє'] != 0:
          р['коеф_варіації'] = р['ст_відхилення'] / р['середнє']
      except Exception:
        pass

    return р


for c in МОЇ:
    print(f'--- {c}  ({ПАСПОРТ[c][0]})')
    print('   ', допустима_статистика(c, ПАСПОРТ[c][0]))
    print()

--- номер і/х  (номінальна)
    {'непорожніх': 121, 'унікальних': 119, 'мода': [1837, 2526], 'частоти': {2526: 2, 1837: 2, '2377/239': 1, '2372/513': 1, '1441/671': 1, '1121/19': 1, '283/674': 1, '2664/348': 1, '1806/185': 1, '1717/25': 1, '3213/476': 1, '2831/393': 1, '2998/135': 1, '2673/681': 1, '1731/280': 1, '2612/542': 1, '3072/418': 1, '2592/430': 1, '1276/596': 1, '2501/673': 1, '612/177': 1, '2573/354': 1, '1762/555': 1, 2320: 1, '2996/84': 1, 2252: 1, '2136/224': 1, '3018/419': 1, '391/531': 1, '2129/293': 1, '1738/55': 1, '2806/426': 1, '1545/172': 1, '144/570': 1, '2813/90': 1, '1324/147': 1, '2848/258': 1, '1006/224': 1, 3304: 1, '1662/15': 1, 3233: 1, 2310: 1, 393: 1, 492: 1, 2455: 1, 296: 1, 3379: 1, 893: 1, 1568: 1, 75: 1, 1803: 1, 416: 1, 1416: 1, '1858/496': 1, '3260/324': 1, '1102/336': 1, '1056/44': 1, '687/376': 1, '2456/355': 1, '1561/701': 1, '79/374': 1, '114/736': 1, '2926/273': 1, '1888/115': 1, '497/721': 1, '3033/34': 1, '2371/432': 1, '1124/483': 1, '1601/8

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [11]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
номер і/х,121.0,119.0,2526.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Стать,121,4,ж,56,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ступінь КТ,42,12,середній,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Дата розтину,61,NaN,NaN,NaN,2022-04-06 13:22:37.377049088,2021-01-06 00:00:00,2022-03-29 00:00:00,2022-04-23 00:00:00,2022-05-07 00:00:00,2022-06-22 00:00:00,NaN
spO2 без дот,120.0,NaN,NaN,NaN,82.049167,0.9,78.75,84.0,89.0,98.0,12.208249
Фібрин1,115.0,87.0,6.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
СРБ2,83.0,NaN,NaN,NaN,10.910843,1.0,6.0,8.0,12.0,72.0,9.798768
ФЕРИТ1,2.0,NaN,NaN,NaN,555.05,271.1,413.075,555.05,697.025,839.0,401.565941


In [12]:
БЕЗГЛУЗДІ = [
    # ('стовпець', 'яка статистика', значення, 'чому це число нічого не означає'),
    ('Дата розтину', 'mean', '2022-04-06 13:22:37.377049088', 'Календарна дата належить до інтервальної шкали, тому обчислення середнього значення за наносекундами немає сенсу '),
    ('Фібрин1', 'mean', 'NaN', 'Даний стовпець має типу object - тому для нього порахувати mean пандас не може'),
    ('Стать', 'mean', 'NaN', 'Стать належть до номінальної шкали, для неї середнє арифметичне в принципі не існує')

]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,Дата розтину,mean,2022-04-06 13:22:37.377049088,"Календарна дата належить до інтервальної шкали, тому обч..."
1,Фібрин1,mean,NaN,Даний стовпець має типу object - тому для нього порахува...
2,Стать,mean,NaN,"Стать належть до номінальної шкали, для неї середнє ариф..."


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [13]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: Гемогл1 × Гематокрит1

Гемогл1: шкала — не у вашому списку
Гематокрит1: шкала — не у вашому списку

  Гемогл1  Гематокрит1
0     157         48.0
1     145         42.2
2     137         40.0
3      87         22.0
4     126         40.0
5     134         39.0


In [14]:
ОБРАНА_МІРА = 'кореляція Пірсона'        # напишіть назву міри
ЧОМУ_САМЕ_ВОНА = 'Обидві змінні є неперервними кількісними ознаками, виміряними у шкалі відношень. Для таких шкал визначені операції віднімання, ділення та обчислення середнього арифметичного і стандартного відхилення, що є математичною базою для коефіцієнта кореляції Пірсона. Крім того ці дві шкали - це шкали відношень тому для них можна застовувати кореляцію Пірсона'     # одне-два речення

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

# ваш код: обчисліть обрану міру
data = df[['Гемогл1', 'Гематокрит1']].copy()
data['Гемогл1'] = pd.to_numeric(data['Гемогл1'], errors='coerce')
data['Гематокрит1'] = pd.to_numeric(data['Гематокрит1'], errors='coerce')
data = data.dropna()

r = data['Гемогл1'].corr(data['Гематокрит1'], method='pearson')
print(f"Кореляція Пірсона: {r:.4f}")

Кореляція Пірсона: 0.7807


In [15]:
# Тепер — контрприклад: застосуйте кореляцію Пірсона незалежно від шкал
# і поясніть, чому отримане число не можна використовувати.
data = df[['Стать', 'Дата розтину']].copy()
data['Стать'] = pd.to_numeric(data['Стать'], errors='coerce')
data['Дата розтину'] = pd.to_numeric(data['Дата розтину'], errors='coerce')
data.dropna()

pearson = data['Стать'].corr(data['Дата розтину'], method='pearson')
print(f"Кореляція Пірсона: {pearson:.4f}")

Кореляція Пірсона: nan


#### Кореляція Пірсона для номінальних ознак у нашому випадку (Стать) не можна використати оскільки вони не є числовими, їх треба спершу перевести в числові. Дата розтину позначає лише календарний момент проведення патологоанатомічного дослідження. Вона не є біологічним параметром організму

# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [16]:
ПІДГОТОВКА = {
    # 'стовпець': 'спосіб і коротке обґрунтування',
    'номер і/х': 'Індексація - це суто технічний ідентифікатор без числового змісту.',
    'Стать': 'One-hot кодування після зведення синонімів до 2 категорій; для номінальної шкали порядок не вводиться.',
    'ступінь КТ': 'Стандартизація одруківок і рангове кодування 1..5; зберігає монотонний порядок тяжкості.',
    'Дата розтину': 'Різниця з датою поступлення; переводить інтервальну дату в тривалість у шкалі відношень.',
    'spO2 без дот': 'Виправлення викиду 0.9 на 90; усуває грубу помилку введення відсотка сатурації.',
    'Фібрин1': 'Очищення від зайвих крапок і приведення до float; повертає числовий тип шкалі відношень.',
    'СРБ2': 'Приведення до float та аналіз пропусків; є коректною неперервною шкалою відношень.',
    'ФЕРИТ1': 'Обробка великої частки пропусків; є коректною кількісною концентрацією у шкалі відношень.'
}
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,номер і/х,номінальна,Індексація - це суто технічний ідентифікатор без числово...
1,Стать,номінальна,One-hot кодування після зведення синонімів до 2 категорі...
2,ступінь КТ,порядкова,Стандартизація одруківок і рангове кодування 1..5; збері...
3,Дата розтину,інтервальна,Різниця з датою поступлення; переводить інтервальну дату...
4,spO2 без дот,абсолютна,Виправлення викиду 0.9 на 90; усуває грубу помилку введе...
5,Фібрин1,відношень,Очищення від зайвих крапок і приведення до float; поверт...
6,СРБ2,відношень,Приведення до float та аналіз пропусків; є коректною неп...
7,ФЕРИТ1,відношень,Обробка великої частки пропусків; є коректною кількісною...


In [17]:
# ваш код: реалізуйте підготовку для трьох стовпців різних шкал
# Стать
mapping_gender = {'ч': 'ч', 'чол': 'ч', 'ж': 'ж', 'жін': 'ж'}
gender_clear = df['Стать'].astype(str).str.strip().str.lower().map(mapping_gender)

df_gender_ohe = pd.get_dummies(gender_clear, prefix='Стать', drop_first=True, dtype=int)
df = pd.concat([df, df_gender_ohe], axis=1)

# Дата розтину
df['дні_до_розтину'] = (pd.to_datetime(df['Дата розтину']) - pd.to_datetime(df['Поступлення'])).dt.days

# ступінь КТ
clear_kt = {
    'легка': 'легкий',
    'легкий': 'легкий',
    'сердній': 'середній',
    'серед ступ': 'середній',
    'серед ступеня': 'середній',
    'середній': 'середній',
    'важкий': 'важкий',
    'вкрай важкий': 'вкрай важкий',
    'критичний': 'критичний'
}

kt_clear = (
    df['ступінь КТ']
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .map(clear_kt)
)


rang_kt = {
    'легкий': 1,
    'середній': 2,
    'важкий': 3,
    'вкрай важкий': 4,
    'критичний': 5
}

df['ступінь_КТ_ранг'] = kt_clear.map(rang_kt)

**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [18]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [19]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [20]:
ВИПРАВЛЕННЯ = [
    # (розділ, 'що не так', 'чому число неправильне', 'як правильно'),
    (1, 'Cереднє арифметичне для бінарної якісної шкали', 'Колонка помер/жив є номінальною бінарною міткою, тому середнє 1.50 не є серединою між станами, а лише фактом умовного кодування 1 і 2.', 'Рахувати відсоткові частки (частоти) виживання та летальності через value_counts(normalize=True).'),
    (2, 'Підрахували просто унікальні значення без стандартизації тексту', 'Число 4 виникло через різне скорочення двох статей', 'Звести синоніми до єдиного вигляду (ч та ж) за допомогою map, після чого df[Стать].nunique() дасть 2'),
    (3, 'Середнє за абетковими кодами порядкової шкали', 'Текстові описи закодували номерами за алфавітним сортуванням, тому середнє 5.9 не має ні фізичного порядку, ні сенсу середнього рівня', 'Очистити одруківки, відсортувати градації за клінічною тяжкістю та вказувати медіану чи моду'),
    (4, 'Множення на 100 величин, що вже були у відсотках', 'Значення у стовпці КТ уже записані як відсотки (наприклад, 40 це 40%), тому множення на 100 дає помилкове спотворення у 100 разів', 'Обчислювати описові статистики напряму для числового стовпця без зайвого домноження на 100'),
    (5, 'Включення грубої помилки введення у розрахунок середнього', 'Значення мінімуму 0.9% фізично неможливе для живої людини; це запис 90%, що занизив середню сатурацію для всієї групи', 'Виправити 0.9 на 90.0 або замінити на np.nan перед обчисленням середнього та квантилів'),
    (6, 'Коефіцієнт варіації для дат відліком від початку епохи', 'Дата є інтервальною шкалою без абсолютного нуля; ділення стандартного відхилення на середнє в днях від 1970 року дає безглуздий штучний малий відсоток', 'Відмовитися від CV для дат або рахувати розкид у шкалі відношень (тривалість між госпіталізаціями чи дні до виписки).'),
    (7, 'Кореляція Пірсона для якісної номінальної шкали', 'Кореляція Пірсона не застосовується до номінальних категорій статі, а значення 0.177 залежить лише від штучно призначених кодів 0 та 1.', 'Використовувати таблицю спряженості pd.crosstab, критерій хі-квадрат (chi2_contingency) та V-Крамера.'),
    (8, 'Min-max нормалізація для порожніх/константних колонок', 'Стовпець ФЕРИТ2 містить лише пропуски або сталі значення; ділення (max - min) викликає ділення на нуль і заповнює все значеннями NaN.', 'Перевіряти стовпці на відсутність варіації (nunique > 1) та частку пропусків перед застосуванням MinMaxScaler.'),
    (9, 'Видалення всіх рядків із пропусками через dropna()', 'У таблиці 77 стовпців, і в кожному рядку знайдеться хоча б один пропуск; виклик dropna() знищив усі 121 спостереження до 0 рядків.', 'Видаляти лише стовпці з високим відсотком пропусків, а для решти застосовувати стратегії імпутації (fillna/SimpleImputer).'),
    (10, 'Кореляція з датою розтину (витік цільової ознаки)', 'Дата розтину заповнена лише для померлих; факт наявності запису і є наслідком смерті (Data Leakage), тому r = 1.00 тривіальний і непридатний.', 'Вилучити стовпець розтину зі списку предикторів летальності, оскільки він стає відомим лише після смерті.')
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,Cереднє арифметичне для бінарної якісної шкали,"Колонка помер/жив є номінальною бінарною міткою, тому се...",Рахувати відсоткові частки (частоти) виживання та леталь...
1,2,Підрахували просто унікальні значення без стандартизації...,Число 4 виникло через різне скорочення двох статей,Звести синоніми до єдиного вигляду (ч та ж) за допомогою...
2,3,Середнє за абетковими кодами порядкової шкали,Текстові описи закодували номерами за алфавітним сортува...,"Очистити одруківки, відсортувати градації за клінічною т..."
3,4,"Множення на 100 величин, що вже були у відсотках",Значення у стовпці КТ уже записані як відсотки (наприкла...,Обчислювати описові статистики напряму для числового сто...
4,5,Включення грубої помилки введення у розрахунок середнього,Значення мінімуму 0.9% фізично неможливе для живої людин...,Виправити 0.9 на 90.0 або замінити на np.nan перед обчис...
5,6,Коефіцієнт варіації для дат відліком від початку епохи,Дата є інтервальною шкалою без абсолютного нуля; ділення...,Відмовитися від CV для дат або рахувати розкид у шкалі в...
6,7,Кореляція Пірсона для якісної номінальної шкали,Кореляція Пірсона не застосовується до номінальних катег...,"Використовувати таблицю спряженості pd.crosstab, критері..."
7,8,Min-max нормалізація для порожніх/константних колонок,Стовпець ФЕРИТ2 містить лише пропуски або сталі значення...,Перевіряти стовпці на відсутність варіації (nunique > 1)...
8,9,Видалення всіх рядків із пропусками через dropna(),"У таблиці 77 стовпців, і в кожному рядку знайдеться хоча...","Видаляти лише стовпці з високим відсотком пропусків, а д..."
9,10,Кореляція з датою розтину (витік цільової ознаки),Дата розтину заповнена лише для померлих; факт наявності...,Вилучити стовпець розтину зі списку предикторів летально...


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [21]:
# ваш код: три правильні обчислення замість хибних
from scipy.stats import chi2_contingency

wrong_1 = df['помер/жив'].mean()
correct_1 = df['помер/жив'].map({1: 'помер', 2: 'вижив'}).value_counts(normalize=True) * 100

print(f"Wrong 1: {wrong_1:.2f}")
print("Correct 1:")
print(correct_1.round(1).to_string())
print()

# 2
wrong_2 = df['Стать'].nunique()
sex_map = {'ч': 'ч', 'чол': 'ч', 'ж': 'ж', 'жін': 'ж'}
sex_clean = df['Стать'].astype(str).str.strip().str.lower().map(sex_map)
correct_2 = sex_clean.value_counts()

print(f"Wrong 2: {wrong_2}")
print(f"Correct 2: {sex_clean.nunique()} values")
print(correct_2.to_string())
print()

# 7
sex_code = df['Стать'].astype(str).str.strip().str[0].map({'ч': 0, 'ж': 1})
target_death = (df['помер/жив'] == 1).astype(int)
wrong_7 = sex_code.corr(target_death)

outcome_labels = df['помер/жив'].map({1: 'помер', 2: 'вижив'})
contingency_table = pd.crosstab(sex_clean, outcome_labels)
chi2_stat, p_val, _, _ = chi2_contingency(contingency_table)

print(f"Wrong 7: {wrong_7:.3f}")
print("Correct 7:")
print(contingency_table)
print(f"Chi2: {chi2_stat:.3f}, p-value: {p_val:.3f}")

Wrong 1: 1.50
Correct 1:
помер/жив
помер    50.4
вижив    49.6

Wrong 2: 4
Correct 2: 2 values
Стать
ж    75
ч    46

Wrong 7: 0.177
Correct 7:
помер/жив  вижив  помер
Стать                  
ж             32     43
ч             28     18
Chi2: 3.086, p-value: 0.079


---
# Етап 7. Висновки та індивідуальне питання

In [22]:
ВИСНОВКИ = [
    # три висновки, кожен — повне речення про ваші дані, а не загальна фраза
    'У досліджуваній вибірці частка летальних випадків склала 50.4% (61 особа), а виписаних — 49.6% (60 осіб), що свідчить про повну збалансованість цільових класів результату лікування.',
    'Після уніфікації текстових скорочень виявлено дві біологічні статі з переважанням жінок (75 пацієнток, або 62%) над чоловіками (46 пацієнтів, або 38%).',
    'Аналіз таблиці спряженості за критерієм хі-квадрат не виявив статистично значущого зв’язку між статтю пацієнта та летальним наслідком госпіталізації (p-value > 0.05).'
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. У досліджуваній вибірці частка летальних випадків склала 50.4% (61 особа), а виписаних — 49.6% (60 осіб), що свідчить про повну збалансованість цільових класів результату лікування.
2. Після уніфікації текстових скорочень виявлено дві біологічні статі з переважанням жінок (75 пацієнток, або 62%) над чоловіками (46 пацієнтів, або 38%).
3. Аналіз таблиці спряженості за критерієм хі-квадрат не виявив статистично значущого зв’язку між статтю пацієнта та летальним наслідком госпіталізації (p-value > 0.05).


In [23]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()
ВІДПОВІДЬ = 'Стовпець КТ поєднує дроби 0.3 і текст 30%. Наївне середнє за допомогою функції pd.to_numeriс(..., errors=coerce) перетворить текст на NaN і втратить усі ці рядки замість їхнього зведення до єдиного масштабу'
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Знайдіть у файлі стовпець, де одна й та сама величина записана у двох несумісних кодуваннях. Скільки спостережень втратить наївне обчислення середнього?

Стовпець КТ поєднує дроби 0.3 і текст 30%. Наївне середнє за допомогою функції pd.to_numeriс(..., errors=coerce) перетворить текст на NaN і втратить усі ці рядки замість їхнього зведення до єдиного масштабу


---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [24]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'текст, перероблено',
    'етап 2 — допустимі статистики':   'код, перевірено',
    'етап 3 — безглузді числа':        'текст, перероблено',
    'етап 4 — міра зв’язку':           'синтаксис',
    'етап 5 — підготовка до моделі':   'код, перевірено',
    'етап 6 — виправлення аналізу':    'текст, як є',
    'етап 7 — висновки й відповідь':   'текст, перероблено',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = 'Доводилося перевіряти чи правильно він генерував віповідь на питання і місцями доводилося його виправляти. Перевіряв також чи правильно він класифікував шкали вимірювання для колонок. '
ПІДТВЕРДЖУЮ = True     # я можу пояснити кожен рядок свого коду
                        # і внести в нього зміни на вимогу під час захисту

In [25]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 18 готова до захисту.
Фарина Арсеній, ШІ-31


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.